In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient

In [0]:
%sql
/*ALTER TABLE 3dt2ndteam5.cwe.cwe_weaknesses
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);*/

In [0]:
import json

ENDPOINT_NAME = "cwe-weaknesses-vector-search"  # Vector Search endpoint 이름
INDEX_NAME = "3dt2ndteam5.cwe.cwe_weaknesses_vs_idx"  # 생성한 인덱스 이름

RETURN_COLUMNS = [
    "weakness_id",
    "title",
    "description",
    "extended_description",
    "potential_mitigations",
    "likelihood_of_exploit",
]

QUERY_TEXT = "Improper input validation can lead to SQL injection."

client = VectorSearchClient()
index = client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

# 인덱스가 준비될 때까지 대기
index.wait_until_ready(wait_for_updates=True)

# Delta Sync + TRIGGERED 모드면 필요 시 동기화 후 검색
# index.sync()
# index.wait_until_ready(wait_for_updates=True)

results = index.similarity_search(
    query_text=QUERY_TEXT,      # 임베딩 컴퓨트 인덱스는 query_text 사용
    columns=RETURN_COLUMNS,
    num_results=5,
    query_type="HYBRID",        # "ANN"으로 바꿔도 됨
)

print(json.dumps(results, ensure_ascii=False, indent=2))


In [0]:
import re
from databricks.vector_search.client import VectorSearchClient

endpoint = "cwe-vector-endpoint"
index_name = "3dt2ndteam5.cwe.cwe_weaknesses_vs_idx"
q = "what is the content of CWE-1004?"

client = VectorSearchClient()
index = client.get_index(endpoint_name=endpoint, index_name=index_name)

d = index.describe()
print("source_table:", d.get("delta_sync_index_spec", {}).get("source_table"))
print("pipeline_type:", d.get("delta_sync_index_spec", {}).get("pipeline_type"))

# TRIGGERED면 최신화
if str(d.get("delta_sync_index_spec", {}).get("pipeline_type", "")).upper() == "TRIGGERED":
    index.sync()
    index.wait_until_ready(wait_for_updates=True)

m = re.search(r"CWE-(\d+)", q, re.I)
if m:
    cwe_id = m.group(1)
    results = index.similarity_search(
        query_text=f"CWE-{cwe_id}",
        query_type="HYBRID",
        columns=["weakness_id", "title", "description"],
        filters={"weakness_id": cwe_id},  # ID 정확 필터
        num_results=5
    )
else:
    results = index.similarity_search(
        query_text=q,
        query_type="HYBRID",
        columns=["weakness_id", "title"],
        filters={"title NOT LIKE": "DEPRECATED"},  # deprecated 제외
        num_results=10
    )

print(results["result"]["data_array"])


In [0]:
%sql
GRANT USE CATALOG ON CATALOG `3dt2ndteam5` TO `3dt016@msacademy.msai.kr`;
GRANT USE SCHEMA ON SCHEMA `3dt2ndteam5`.`js_vuln_db` TO `3dt016@msacademy.msai.kr`;
GRANT SELECT, MODIFY ON TABLE `3dt2ndteam5`.`js_vuln_db`.`js_cwe` TO `3dt016@msacademy.msai.kr`;

In [0]:
%sql
SHOW TABLES IN `3dt2ndteam5`.`js_vuln_db`;
SELECT * FROM `3dt2ndteam5`.`js_vuln_db`.`js_cwe` LIMIT 1;
